# 🎯 Fine-Tuning with Local Testing Loop

**Iterative Development Workflow:**
1. Fine-tune model in Colab (GPU)
2. Auto-export to XJSON after each epoch
3. Download and test locally with WebGPU runtime
4. Iterate based on local test results
5. Continue training with adjustments

**This notebook includes:**
- Full fine-tuning setup
- Automatic XJSON export on checkpoints
- Google Drive sync for easy downloads
- Training monitoring and logging

In [ ]:
# Install dependencies
!pip install torch transformers datasets accelerate wandb -q
!pip install -U huggingface_hub -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import json
import numpy as np
from pathlib import Path
import os
from tqdm.auto import tqdm

print("✅ Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 1. Mount Google Drive (Optional - for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
output_dir = Path('/content/drive/MyDrive/fractal_models')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"   Output directory: {output_dir}")

## 2. Define Model (same as export notebook)

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=1000, n_embd=128, n_head=4, n_layer=2, max_seq_len=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.n_head = n_head
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len
        
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(max_seq_len, n_embd)
        
        self.layers = nn.ModuleList([
            TransformerBlock(n_embd, n_head) for _ in range(n_layer)
        ])
        
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        for layer in self.layers:
            x = layer(x)
        
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd)
        )
        
    def forward(self, x):
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.ffn(self.ln2(x))
        return x

print("✅ Model architecture defined")

## 3. XJSON Export Function

In [ ]:
def export_to_xjson(model: nn.Module, output_path: str, epoch: int = 0, metrics: dict = None):
    """
    Export model to XJSON with training metadata
    """
    print(f"\n🔄 Exporting checkpoint (epoch {epoch})...")
    
    config = {
        'vocab_size': model.vocab_size,
        'n_embd': model.n_embd,
        'n_head': model.n_head,
        'n_layer': model.n_layer,
        'max_seq_len': model.max_seq_len,
    }
    
    layers = [
        {'type': 'embedding', 'name': 'token_embedding', 'params': {'vocab_size': config['vocab_size'], 'embedding_dim': config['n_embd']}},
        {'type': 'embedding', 'name': 'position_embedding', 'params': {'max_positions': config['max_seq_len'], 'embedding_dim': config['n_embd']}}
    ]
    
    for i in range(config['n_layer']):
        layers.append({'type': 'transformer_block', 'name': f'layer_{i}', 'params': {'n_embd': config['n_embd'], 'n_head': config['n_head']}})
    
    layers.extend([
        {'type': 'layer_norm', 'name': 'final_norm', 'params': {'n_embd': config['n_embd']}},
        {'type': 'linear', 'name': 'lm_head', 'params': {'in_features': config['n_embd'], 'out_features': config['vocab_size']}}
    ])
    
    # Extract weights (with progress bar for large models)
    weights = {}
    state_dict = model.state_dict()
    
    for name, param in tqdm(state_dict.items(), desc="Extracting weights"):
        weight_data = param.cpu().detach().numpy()
        weights[name] = {
            'data': weight_data.tolist(),
            'shape': list(weight_data.shape),
            'dtype': 'float32'
        }
    
    xjson = {
        'fractal_runtime': '1.0',
        'model_type': 'transformer',
        'config': config,
        'architecture': 'transformer',
        'layers': layers,
        'weights': weights,
        'metadata': {
            'name': f'finetuned-epoch-{epoch}',
            'version': '0.1.0',
            'framework': 'pytorch',
            'training_epoch': epoch,
            'total_params': sum(p.numel() for p in model.parameters()),
            'compatible_with': ['@fractal/gpu-runtime', '@xjson/xjson-server']
        }
    }
    
    # Add training metrics if provided
    if metrics:
        xjson['metadata']['training_metrics'] = metrics
    
    with open(output_path, 'w') as f:
        json.dump(xjson, f)
    
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ Exported to {output_path} ({size_mb:.2f} MB)")
    
    return output_path

print("✅ Export function ready")

## 4. Create Dummy Dataset (Replace with your data)

In [ ]:
class DummyTextDataset(Dataset):
    """Replace this with your actual dataset"""
    def __init__(self, vocab_size=1000, seq_len=64, num_samples=10000):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.num_samples = num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Random tokens (replace with real tokenized text)
        tokens = torch.randint(0, self.vocab_size, (self.seq_len,))
        return {'input_ids': tokens, 'labels': tokens}

# Create datasets
train_dataset = DummyTextDataset(vocab_size=1000, seq_len=64, num_samples=10000)
val_dataset = DummyTextDataset(vocab_size=1000, seq_len=64, num_samples=1000)

print(f"✅ Dataset created")
print(f"   Training samples: {len(train_dataset):,}")
print(f"   Validation samples: {len(val_dataset):,}")
print("\n⚠️  NOTE: Replace DummyTextDataset with your actual data!")

## 5. Training Loop with Auto-Export

In [ ]:
def train_with_export(
    model,
    train_dataset,
    val_dataset,
    output_dir,
    epochs=5,
    batch_size=32,
    learning_rate=1e-4,
    export_every_epoch=True
):
    """
    Train model with automatic XJSON export after each epoch
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    print("🚀 Starting training...")
    print(f"   Device: {device}")
    print(f"   Batch size: {batch_size}")
    print(f"   Learning rate: {learning_rate}")
    print(f"   Epochs: {epochs}")
    print(f"   Export checkpoints: {export_every_epoch}\n")
    
    training_history = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_steps = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            logits = model(input_ids)
            loss = criterion(logits.view(-1, model.vocab_size), labels.view(-1))
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_steps += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_train_loss = train_loss / train_steps
        
        # Validation
        model.eval()
        val_loss = 0
        val_steps = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                
                logits = model(input_ids)
                loss = criterion(logits.view(-1, model.vocab_size), labels.view(-1))
                
                val_loss += loss.item()
                val_steps += 1
        
        avg_val_loss = val_loss / val_steps
        
        metrics = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss
        }
        training_history.append(metrics)
        
        print(f"\n📊 Epoch {epoch+1} Results:")
        print(f"   Train Loss: {avg_train_loss:.4f}")
        print(f"   Val Loss:   {avg_val_loss:.4f}")
        
        # Export to XJSON
        if export_every_epoch:
            checkpoint_path = output_dir / f'checkpoint_epoch_{epoch+1}.xjson'
            export_to_xjson(model, str(checkpoint_path), epoch=epoch+1, metrics=metrics)
            print(f"   📥 Download from: {checkpoint_path}")
        
        # Save PyTorch checkpoint too
        torch_checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss
        }
        torch.save(torch_checkpoint, output_dir / f'checkpoint_epoch_{epoch+1}.pt')
    
    print("\n🎉 Training complete!")
    
    # Final export
    final_path = output_dir / 'final_model.xjson'
    export_to_xjson(model, str(final_path), epoch=epochs, metrics=training_history[-1])
    
    return model, training_history

print("✅ Training function ready")

## 6. Initialize Model and Start Training

In [ ]:
# Create model
model = SimpleTransformer(
    vocab_size=1000,
    n_embd=128,
    n_head=4,
    n_layer=2,
    max_seq_len=256
)

print(f"✅ Model initialized: {sum(p.numel() for p in model.parameters()):,} parameters")

# Start training
trained_model, history = train_with_export(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    output_dir=output_dir,
    epochs=5,
    batch_size=32,
    learning_rate=1e-4,
    export_every_epoch=True
)

## 7. Download Checkpoints for Local Testing

In [ ]:
# List all exported models
import glob

xjson_files = sorted(glob.glob(str(output_dir / '*.xjson')))

print("📦 Available XJSON Exports:")
for i, path in enumerate(xjson_files, 1):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"   {i}. {Path(path).name} ({size_mb:.2f} MB)")

# Download latest checkpoint
if xjson_files:
    latest = xjson_files[-1]
    print(f"\n📥 Downloading latest: {Path(latest).name}")
    
    from google.colab import files
    files.download(latest)
    
    print("\n✅ Download started!")
    print("\n📝 Next steps:")
    print("   1. Save the downloaded file to FRACTAL-GPU-RUNTIME directory")
    print("   2. Test locally:")
    print(f"      node examples/load-xjson.js {Path(latest).name}")
    print("   3. Run inference and validate")
    print("   4. Come back here to continue training if needed")

## 8. Training Visualization

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, 'b-', label='Train Loss', marker='o')
plt.plot(epochs, val_losses, 'r-', label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / 'training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training curve saved to Google Drive")

## 9. Resume Training from Checkpoint

In [ ]:
# If you need to resume training
def resume_training(checkpoint_path, additional_epochs=5):
    checkpoint = torch.load(checkpoint_path)
    
    # Recreate model
    model = SimpleTransformer(
        vocab_size=1000,
        n_embd=128,
        n_head=4,
        n_layer=2,
        max_seq_len=256
    )
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"✅ Resumed from epoch {checkpoint['epoch']}")
    print(f"   Previous train loss: {checkpoint['train_loss']:.4f}")
    print(f"   Previous val loss: {checkpoint['val_loss']:.4f}")
    
    # Continue training
    trained_model, new_history = train_with_export(
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        output_dir=output_dir,
        epochs=additional_epochs,
        batch_size=32,
        learning_rate=1e-4,
        export_every_epoch=True
    )
    
    return trained_model, new_history

# Uncomment to resume:
# model, history = resume_training(output_dir / 'checkpoint_epoch_5.pt', additional_epochs=5)

## 🔄 Complete Workflow

### In Colab (This Notebook):
1. ✅ Fine-tune model
2. ✅ Auto-export to XJSON after each epoch
3. ✅ Download checkpoints

### On Your Local Machine:
```bash
# Test the exported model
node examples/load-xjson.js checkpoint_epoch_5.xjson

# Or use in your own code
const runtime = new FractalRuntime();
await runtime.init();
const model = await runtime.loadModel('./checkpoint_epoch_5.xjson', 'finetuned');
const result = await runtime.infer('finetuned', tokens);
```

### Back to Colab:
4. Resume training if needed
5. Export final model
6. Deploy with @xjson/klh-orchestrator

## 📚 Resources

- [FRACTAL GPU Runtime](https://github.com/cannaseedus-bot/FRACTAL-GPU-RUNTIME)
- [Quick Start Guide](../QUICKSTART.md)
- [@xjson packages](https://www.npmjs.com/settings/xjson/packages)